In [1]:
from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd
from rasterio.merge import merge

def gatherData(dataset, year, city, aoi_geodf, isGroundTruth = False):
    truthString = "GroundTruth" if isGroundTruth else "InputData"
    print(f'Gathering {truthString} {dataset} for {year} in {city}.')
    bandNames = {'B2', 'B3', 'B4', 'B5', 'B6', 'ST_B10'}
    if isGroundTruth:
        bandNames = {'ST_B10'}
    includeMetadata = True
    #%%
    # Cell 6: Search for Scenes
    if dataset == 'srtm_v3' and year != 2014:
        return
    for month in range(1, 13):
        print("Starting month", month)
        clear_folder(unprocessed_dir)
        search_payload = createSceneSearchPayload(dataset, aoi_geodf, year, month, 20)
        scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
        pd.json_normalize(scenes['results'])
        if len(scenes['results']) == 0:
            print("Month for scenes empty, skipping...")
            continue
        #%%
        # Cell 7: Collect Entity IDs
        entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]
        #%%
        # Cell 8: Prepare Scene List for Download
        listId = f"{dataset}_{year}_{str(month)}_{'truth' if isGroundTruth else 'notTruth'}"
        scn_list_add_payload = {
            "listId": listId,
            'idField': 'entityId',
            "entityIds": entityIds,
            "datasetName": dataset
        }
        sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)
        #%%
        # Cell 9: Prepare Download Options
        download_opt_payload = {
            "listId": listId,
            "datasetName": dataset,
        }
        products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
        pd.json_normalize(products)
        # print(products)
        #%%
        # Cell 10: Collect Files to Download
        downloads = []
        if 'landsat_ot_c2_l2' in dataset:
            for product in products:
                if product["secondaryDownloads"]:
                    for secDownload in product["secondaryDownloads"]:
                        if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                            downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
                        if includeMetadata and secDownload['displayId'].endswith('_MTL.txt'):
                            downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
        elif 'srtm_v3' in dataset:
            for product in products:
                if product["bulkAvailable"] and product["entityId"] and product["id"]:
                    downloads.append({"entityId": product["entityId"], "productId": product["id"]})
        elif 'ccdc_v1_3' in dataset:
            for product in products:
                if product["bulkAvailable"] and product["entityId"] and product["id"]:
                    downloads.append({"entityId": product["entityId"], "productId": product["id"]})
        #%%
        # Cell 11: Submit Download Request
        download_req_payload = {
            "downloads": downloads,
            "label": listId
        }
        download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)
        #%%
        # print(download_request_results)
        #%%
        # Cell 12: Download Files
        if dataset == 'landsat_ot_c2_l2':
            results = download_request_results['availableDownloads']
            for result in results:
                runDownload(threads, result['url'])
        elif dataset == 'srtm_v3':
            results = download_request_results['preparingDownloads']
            for result in results:
                runDownload(threads, result['url'])
        else:
            results = download_request_results['preparingDownloads']
            for result in results:
                runDownload(threads, result['url'])
        # print("before join")
        for t in threads:
            t.join()
        # print("after join")
        for file in os.listdir('./Unprocessed'):
            if file.endswith('.tar'):
                try: 
                    extract_specific_files('./Unprocessed/' + file, './Unprocessed')
                except:
                    print(f'Error: Could not extract file {file}')
        #%%
        # Cell 13: Clean Up
        remove_scnlst_payload = {"listId": listId}
        sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)
        #%%
        # Cell 15: Verify Downloads
        import rasterio
        from shapely.geometry import box
        
        print('Unprocessed Dir:', os.listdir(unprocessed_dir))
        tifs = [[],[],[],[],[],[],[],[]] # B2 B3 B4 B5 B6 B10 LCPRI v3 
        for file in os.listdir(unprocessed_dir):
            if file.endswith(".TIF") or file.endswith(".tif"):
                if '1arc_v3' in file:
                    tifs[7].append(file)
                elif 'LCMAP' in file:
                    if 'LCPRI' in file:
                        tifs[6].append(file)
                else:
                    date, band, coordinate = getMetaFromLandsatTIRs(file)
                    if band == 'B10':
                        tifs[5].append(file)
                    elif band == 'B2':
                        tifs[0].append(file)
                    elif band == 'B3':
                        tifs[1].append(file)
                    elif band == 'B4':
                        tifs[2].append(file)
                    elif band == 'B5':
                        tifs[3].append(file)
                    elif band == 'B6':
                        tifs[4].append(file)
        for tifs_to_merge in tifs:
            if tifs_to_merge:
                for tif in tifs_to_merge:                
                    temp_path = ".temp"  # Temporary file path
                    with rioxarray.open_rasterio(unprocessed_dir + '/' + tif) as raster:
                        raster = raster.rio.reproject("EPSG:4326")
                        raster.rio.to_raster(temp_path, driver="GTiff")
                    os.replace(temp_path, unprocessed_dir + '/' + tif)
                    if os.path.exists(temp_path):
                        os.remove(temp_path)
        for tifs_to_merge in tifs:
            if tifs_to_merge:
                output_path = unprocessed_dir + '/' + tifs_to_merge[0]  # Overwrite the first file in the list
                with rasterio.open(output_path) as src:
                    meta = src.meta
                sources = [rasterio.open(unprocessed_dir + '/' + tif) for tif in tifs_to_merge]
                merged_array, merged_transform = merge(sources)
                meta.update({
                    "driver": "GTiff",
                    "height": merged_array.shape[1],
                    "width": merged_array.shape[2],
                    "transform": merged_transform
                })
                for source in sources:
                    source.close()
                with rasterio.open(output_path, "w", **meta) as dest:
                    dest.write(merged_array)
                print(f"Overwritten: {output_path}")
        usableTIFFs = []
        for tifsToMerge in tifs:
            if len(tifsToMerge) == 0:
                continue
            tif_path = unprocessed_dir + '/' + tifsToMerge[0]
            with rasterio.open(tif_path) as src:
                # Reproject shapefile to match TIF file CRS
                aoi_geodf_proj = aoi_geodf.to_crs(src.crs)
                tif_bounds = box(*src.bounds)
                
                # Check containment
                for idx, geom in enumerate(aoi_geodf_proj.geometry):
                    if tif_bounds.contains(geom):
                        usableTIFFs.append(tifsToMerge[0])
                        print(f"Polygon {city} is fully inside {tifsToMerge[0]}")
                    else:
                        print(f"Polygon {city} is NOT fully inside {tifsToMerge[0]}")
        print('Usable Tiffs:', usableTIFFs)
        #%%
        goodCoordinates = clipUnprocessedRasters(usableTIFFs, aoi_geodf_proj)
        #%%
        for file in os.listdir(unprocessed_dir):
            if ".txt" in file or "Clipped_" in file:
                if '1arc_v3' in file:
                    moveToRaw(file, 'DEM', f'{year}-01-01', city)
                    continue
                if 'LCMAP' in file:
                    if 'LCPRI' in file:
                        moveToRaw(file, 'Land_Cover', f'{year}-01-01', city)
                    continue
                date, band, coordinate = getMetaFromLandsatTIRs(file)
                if coordinate not in goodCoordinates:
                    continue
                print(date, city, band)
                if band == 'B10':
                    if isGroundTruth:
                        moveToRaw(file, 'labelLST', date, city)
                        continue
                    else:
                        moveToRaw(file, 'LST', date, city)
                if band == 'B2':
                    moveToRaw(file, 'Albedo', date, city)
                if band == 'B3':
                    moveToRaw(file, 'NDWI', date, city)
                if band == 'B4':
                    moveToRaw(file, 'Albedo', date, city)
                    moveToRaw(file, 'NDVI', date, city)
                if band == 'B5':
                    moveToRaw(file, 'Albedo', date, city)
                    moveToRaw(file, 'NDVI', date, city)
                    moveToRaw(file, 'NDWI', date, city)
                if band == 'B6':
                    moveToRaw(file, 'Albedo', date, city)
                if band == 'MTL':
                    moveToRaw(file, 'Albedo', date, city)
                    moveToRaw(file, 'LST', date, city)
                    moveToRaw(file, 'NDVI', date, city)
                    moveToRaw(file, 'NDWI', date, city)    
        #%%
        print('Finished moving')
        if dataset == 'ccdc_v1_3' or dataset == 'srtm_v3':
            break
    truthString = "InputData"
    if isGroundTruth:
        truthString = "GroundTruth"
    with open('progress.txt', "a") as file:
        file.write(str(truthString + ":" + str(city) + ":" + str(year) + ":" + dataset + "\n"))
    print('progress written for', truthString, city, year, dataset)

Directory './Unprocessed' created successfully.
Directory './Data' already exists.
Directory './RawClippedRasters' created successfully.
Directory './Data/LST' already exists.
Directory './Data/NDVI' already exists.
Directory './Data/NDWI' already exists.
Directory './Data/Land_Cover' already exists.
Directory './Data/Albedo' already exists.
Directory './Data/DEM' already exists.
Directory './Data/labelLST' already exists.
Directory './RawClippedRasters/LST' created successfully.
Directory './RawClippedRasters/NDVI' created successfully.
Directory './RawClippedRasters/NDWI' created successfully.
Directory './RawClippedRasters/Land_Cover' created successfully.
Directory './RawClippedRasters/Albedo' created successfully.
Directory './RawClippedRasters/DEM' created successfully.
Directory './RawClippedRasters/labelLST' created successfully.
Logging in...


Login Successful, API Key Received!


In [ ]:
#TODO: Run it again to make sure progress is skipped...
datasets = ['landsat_ot_c2_l2', 'srtm_v3', 'ccdc_v1_3', 'landsat_ot_c2_l2'] 
'''['nlcd_collection_lndcov']'''
years = [year for year in range(2013, 2022)]

# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapeGroundTruth_folder = "./Data/label_shp/"
cities = []
citiesT = []
aoi_geodfs = []
aoi_geodfsTruth = []
for file in os.listdir(shapefile_folder):
    if file.endswith(".shp"):
        cities.append(file.replace('Polygon_', '').replace('.shp', ''))
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        aoi_geodfs.append(aoi_geodf)
for file in os.listdir(shapeGroundTruth_folder):
    if file.endswith(".shp"):
        citiesT.append(file.replace('Polygon_', '').replace('.shp', ''))
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        aoi_geodfsTruth.append(aoi_geodf)
print("Shapefiles loaded successfully.")
import traceback
for j, dataset in enumerate(datasets):
    for year in years:
        i = 0
        while i < len(cities):
            try:
                clear_folder(unprocessed_dir)
                assert len(os.listdir(unprocessed_dir)) == 0, "Unprocessed directory is not empty."
                isGroundTruth = j == len(datasets) - 1
                truthString = "GroundTruth" if isGroundTruth else "InputData"
                city, aoi_geodf = citiesT[i] if isGroundTruth else cities[i], aoi_geodfsTruth[i] if isGroundTruth else aoi_geodfs[i]
                if os.path.exists('progress.txt'):
                    with open('progress.txt', 'r') as file:
                        progress = [line.split(':') for line in file.read().strip().split('\n')]
                    if any(truthString == instance[0] and city == instance[1] and str(year) == instance[2] and dataset == instance[3] for instance in progress):
                        print(f"{truthString}, {city}, {year}, {dataset} was gathered in the past.")
                    else:
                        gatherData(dataset, year, city, aoi_geodf, isGroundTruth)
                i += 1
            except Exception as e:
                print("An exception occurred:")
                print(f"Exception: {e}")
                traceback.print_exc()  # Print the full stack trace
                time.sleep(60)
print("Gathered data successfully.")

Shapefiles loaded successfully.
InputData, Abilene_TX, 2013, landsat_ot_c2_l2 was gathered in the past.
Gathering InputData landsat_ot_c2_l2 for 2013 in Albuquerque_NM.
Starting month 1
Month for scenes empty, skipping...
Starting month 2
Month for scenes empty, skipping...
Starting month 3
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_MTL.txt...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_SR_B2.TIF...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_SR_B3.TIF...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_SR_B5.TIF...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_SR_B4.TIF...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_SR_B6.TIF...
Downloading: LC08_L2SP_033036_20130318_20200913_02_T1_ST_B10.TIF...
Unprocessed Dir: ['LC08_L2SP_033036_20130318_20200913_02_T1_MTL.txt', 'LC08_L2SP_033036_20130318_20200913_02_T1_SR_B2.TIF', 'LC08_L2SP_033036_20130318_20200913_02_T1_SR_B3.TIF', 'LC08_L2SP_033036_20130318_20200913_02_T1_SR_B4.TIF', 'LC08_L2

In [ ]:
'''def calculate_ndvi(band4_path, band5_path, output_path):
    # Read Band 4 (Red) and Band 5 (NIR)
    with rasterio.open(band4_path) as band4:
        red = band4.read(1).astype('float32') / 10000  # Scale surface reflectance
        profile = band4.profile  # Keep metadata

    with rasterio.open(band5_path) as band5:
        nir = band5.read(1).astype('float32') / 10000  # Scale surface reflectance

    # NDVI calculation
    ndvi = (nir - red) / (nir + red + 1e-10)

    # Update metadata for NDVI
    profile.update(dtype='float32', count=1)

    # Write NDVI to output raster
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(ndvi, 1)
    print("NDVI calculated successfully.")
    
def calculate_ndwi(band3_path, band5_path, output_path):
    # Read Band 3 (Green) and Band 5 (NIR)
    with rasterio.open(band3_path) as band3:
        green = band3.read(1).astype('float32') / 10000  # Scale surface reflectance
        profile = band3.profile  # Keep metadata

    with rasterio.open(band5_path) as band5:
        nir = band5.read(1).astype('float32') / 10000  # Scale surface reflectance

    # NDWI calculation
    ndwi = (green - nir) / (green + nir + 1e-10)  # Small epsilon to avoid division by zero

    # Update metadata for NDWI
    profile.update(dtype='float32', count=1)

    # Write NDWI to output raster
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(ndwi, 1)
    print("NDWI calculated successfully.")

invalidPaths = []
for dataType in ['NDVI', 'DEM', 'Land_Cover', 'Albedo', 'NDWI']:
    baseFolder = raw_dir + '/' + dataType
    for dateFolder in os.listdir(baseFolder):
        for cityFolder in os.listdir(baseFolder + '/' + dateFolder):
            if dateFolder + '/' + cityFolder in invalidPaths:
                print("Invalid from other data", dateFolder + '/' + cityFolder, end="...")
                continue
            bandFolder = baseFolder + '/' + dateFolder + '/' + cityFolder
            print("Calculating", bandFolder, end=":")
            if dataType == 'NDVI':
                band4 = [f for f in os.listdir(bandFolder) if '_B4.TIF' in f]
                band5 = [f for f in os.listdir(bandFolder) if '_B5.TIF' in f]
                if len(band4) == 0 or len(band5) == 0:
                    print("invalid NDVI", dateFolder + '/' + cityFolder, end="...")
                    invalidPaths.append(dateFolder + '/' + cityFolder)
                    continue
                band4_path = os.path.join(bandFolder, band4[0])
                band5_path = os.path.join(bandFolder, band5[0])
            elif dataType == 'DEM':
                dem = [f for f in os.listdir(bandFolder) if '1arc' in f]
                if len(dem) == 0:
                    print("invalid DEM", dateFolder + '/' + cityFolder, end="...")
                    invalidPaths.append(dateFolder + '/' + cityFolder)
                    continue
                demPath = os.path.join(bandFolder, dem[0])
            elif dataType == 'Land_Cover':
                landCover = [f for f in os.listdir(bandFolder) if 'LCPRI' in f]
                if len(landCover) == 0:
                    print("invalid LandCover", dateFolder + '/' + cityFolder, end="...")
                    invalidPaths.append(dateFolder + '/' + cityFolder)
                    continue
                landCoverPath = os.path.join(bandFolder, landCover[0])
            elif dataType == 'NDWI':
                band3 = [f for f in os.listdir(bandFolder) if '_B3.TIF' in f]
                band5 = [f for f in os.listdir(bandFolder) if '_B5.TIF' in f]
                if len(band3) == 0 or len(band5) == 0:
                    print("invalid NDWI", dateFolder + '/' + cityFolder, end="...")
                    invalidPaths.append(dateFolder + '/' + cityFolder)
                    continue
                band3_path = os.path.join(bandFolder, band3[0])
                band5_path = os.path.join(bandFolder, band5[0])
            dataFolder = './Data/' + dataType + '/' + dateFolder + '/' + cityFolder
            if not os.path.exists(dataFolder):
                os.makedirs(dataFolder)
            outFile = dataType + '_' + dateFolder + '_' + cityFolder + '.TIF'
            if dataType == 'NDVI':
                calculate_ndvi(band4_path, band5_path, dataFolder + "/" + outFile)
            if dataType == 'DEM':
                shutil.copy(demPath, dataFolder + '/' + outFile)
            if dataType == 'Land_Cover':
                shutil.copy(landCoverPath, dataFolder + '/' + outFile)
            if dataType == 'NDWI':
                calculate_ndwi(band3_path, band5_path, dataFolder + "/" + outFile)
   '''         